# ChemicalEntity ↔ Pathway Relation-Wise Merge

Merges Chemical–Pathway triples from Monarch and iBKH; resolves chemical names via
PubChem and pathway names via Reactome; assigns `tail_id_is` based on pathway ID prefix
(Reactome vs KEGG); deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd
import numpy as np
import re

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_PATHWAY/ALL_CHEMICALENTITY_PATHWAY.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Chemical — PubChem

In [2]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem
print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


### 1b. Chemical — DrugBank (fallback for DB-prefixed IDs)

In [3]:
Drugbank = pd.read_csv(DB_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))
print(f"DrugBank dict: {len(Drugbank_dict):,} entries")

DrugBank dict: 16,575 entries


### 1b. Pathway — Reactome (Human only)

In [4]:
Reactome = pd.read_csv(DB_DIR + 'reactome/ReactomePathways.txt', sep='\t', header=None)
Reactome = Reactome[Reactome[2] == 'Homo sapiens']
Reactome_dict = dict(zip(Reactome[0], Reactome[1]))
print(f"Reactome dict (Human): {len(Reactome_dict):,} entries")

Reactome dict (Human): 2,751 entries


In [21]:
## 1c Kegg pathway

	# KEGG:map07051
kegg_pathway = pd.read_csv(DB_DIR + 'kegg/kegg_pathways.txt', sep='\t', header=None)
kegg_pathway[0] = 'KEGG:' + kegg_pathway[0]
kegg_pathway_dict = dict(zip(kegg_pathway[0], kegg_pathway[1]))
kegg_pathway

,0,1
0,KEGG:map01100,Metabolic pathways
1,KEGG:map01110,Biosynthesis of secondary metabolites
2,KEGG:map01120,Microbial metabolism in diverse environments
3,KEGG:map01200,Carbon metabolism
4,KEGG:map01210,2-Oxocarboxylic acid metabolism
...,...,...
580,KEGG:map07035,Prostaglandins
581,KEGG:map07110,Benzoic acid family
582,KEGG:map07112,"1,2-Diphenyl substitution family"
583,KEGG:map07114,Naphthalene family


## 2. Load KG Sources

### 2a. Monarch

In [33]:
Monarch_Chemical_Pathway = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_Chemical_Pathway_Monarch.csv')
Monarch_Chemical_Pathway.columns = Monarch_Chemical_Pathway.columns.str.lower()
Monarch_Chemical_Pathway.rename(columns={'kgsource': 'kg_source', 'tail_name': 'tail_detail_name'}, inplace=True)
print(f"Monarch:  {len(Monarch_Chemical_Pathway):,} rows | columns: {list(Monarch_Chemical_Pathway.columns)}")

Monarch_Chemical_Pathway['kg_type'] = 'Generalised'
Monarch_Chemical_Pathway['kg_source'] = 'Monarch_KG'
Monarch_Chemical_Pathway['species'] = 'Homo species'

Monarch_Chemical_Pathway.head(2)

Monarch:  8,668 rows | columns: ['head', 'tail', 'relation_type', 'relation_source', 'kg_source', 'head_name', 'head_type', 'head_id_is', 'head_taxon', 'head_taxon_label', 'tail_detail_name', 'tail_type', 'tail_id_is', 'tail_taxon', 'tail_taxon_label', 'head_taxon_name', 'tail_taxon_name', 'head_type_clean', 'tail_type_clean', 'relation', 'head_id', 'species_species']


,head,tail,relation_type,relation_source,kg_source,head_name,head_type,head_id_is,head_taxon,head_taxon_label,...,tail_taxon_label,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,head_id,species_species,kg_type,species
0,5461108,R-HSA-8939211,participates_in,infores:reactome,Monarch_KG,ATP(4-),ChemicalEntity,Pubchem,NaN,NaN,...,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,Pathway,ChemicalEntity_Pathway,CHEBI:30616,Homo sapiens_Homo sapiens,Generalised,Homo species
1,5461108,R-HSA-8939243,participates_in,infores:reactome,Monarch_KG,ATP(4-),ChemicalEntity,Pubchem,NaN,NaN,...,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,Pathway,ChemicalEntity_Pathway,CHEBI:30616,Homo sapiens_Homo sapiens,Generalised,Homo species


### 2b. iBKH

Fix swapped `tail_detail_name` columns.

In [35]:
iBKH_Chemical_Pathway = pd.read_csv(PROC_DIR + 'iBKH/Drug_Pathway_ibkh.csv')
iBKH_Chemical_Pathway['Tail_detail_name'] = iBKH_Chemical_Pathway['Tail'].map(kegg_pathway_dict)
iBKH_Chemical_Pathway.columns = iBKH_Chemical_Pathway.columns.str.lower()
# Fix swapped name columns
# iBKH_Chemical_Pathway[['tail_detail_name', 'tail_detail_name.1']] = \
#     iBKH_Chemical_Pathway[['tail_detail_name.1', 'tail_detail_name']]

iBKH_Chemical_Pathway['kg_type'] = 'Generalised'

iBKH_Chemical_Pathway['species'] = 'Homo species'

print(f"iBKH:     {len(iBKH_Chemical_Pathway):,} rows | columns: {list(iBKH_Chemical_Pathway.columns)}")
iBKH_Chemical_Pathway.head(2)

iBKH:     1,670 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'tail_detail_name.1', 'kg_type', 'species']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,tail_detail_name.1,kg_type,species
0,2554,ChemicalEntity_Pathway,KEGG:map00982,ChemicalEntity,Pathway,iBKH,benzo[b][1]benzazepine-11-carboxamide,C1=CC=C2C(=C1)C=CC3=CC=CC=C3N2C(=O)N,Drug metabolism - cytochrome P450,Pubchem,KEGG_ID,NaN,Generalised,Homo species
1,3690,ChemicalEntity_Pathway,KEGG:map00982,ChemicalEntity,Pathway,iBKH,"N,3-bis(2-chloroethyl)-2-oxo-1,3,2lambda5-oxaz...",C1CN(P(=O)(OC1)NCCCl)CCCl,Drug metabolism - cytochrome P450,Pubchem,KEGG_ID,NaN,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [36]:
SOURCE_DFS = [
    ('Monarch_Chemical_Pathway', Monarch_Chemical_Pathway),
    ('iBKH_Chemical_Pathway',    iBKH_Chemical_Pathway),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Chemical_Pathway] ✓ no duplicates
[iBKH_Chemical_Pathway] ✓ no duplicates


In [37]:
# # Edit keep='first'/'last' per DataFrame based on inspection above
# Monarch_Chemical_Pathway = Monarch_Chemical_Pathway.loc[:, ~Monarch_Chemical_Pathway.columns.duplicated(keep='first')]
# iBKH_Chemical_Pathway    = iBKH_Chemical_Pathway.loc[:,    ~iBKH_Chemical_Pathway.columns.duplicated(keep='first')]

# for name, df in SOURCE_DFS:
#     remaining = df.columns[df.columns.duplicated()].tolist()
#     print(f"[{name}] {'✓ clean' if not remaining else '✗ still has dupes: ' + str(remaining)}")

## 4. Align Schemas and Concatenate

In [38]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
final_df['head'] = final_df['head'].astype(str)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 10,338 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,5461108,ChemicalEntity_Pathway,R-HSA-8939211,ChemicalEntity,participates_in,Pathway,Monarch_KG,Pubchem,Reactome,NaN,ESR-mediated signaling,Generalised,Homo species
1,5461108,ChemicalEntity_Pathway,R-HSA-8939243,ChemicalEntity,participates_in,Pathway,Monarch_KG,Pubchem,Reactome,NaN,RUNX1 interacts with co-factors whose precise ...,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [39]:
final_df['head_type'] = 'ChemicalEntity'
final_df['tail_type'] = 'Pathway'
final_df['relation']  = 'ChemicalEntity_Pathway'

# head_id_is: DB-prefix → Drugbank, else Pubchem
final_df['head_id_is'] = np.where(
    final_df['head'].isna(), None,
    np.where(final_df['head'].str.startswith('DB'), 'Drugbank', 'Pubchem')
)

# tail_id_is: R-prefix → Reactome_ID, else KEGG_ID
final_df['tail'] = final_df['tail'].astype(str)
final_df['tail_id_is'] = np.where(
    final_df['tail'].isna(), None,
    np.where(final_df['tail'].str.startswith('R'), 'Reactome_ID', 'KEGG_ID')
)

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'ChemicalEntity_Pathway'}
Unique head_id_is: {'Pubchem'}
Unique tail_id_is: {'KEGG_ID', 'Reactome_ID'}
Unique kg_source: {'Monarch_KG', 'iBKH'}


## 6. Deduplicate

In [40]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
final_df_dedup['head'] = final_df_dedup['head'].astype(str)
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 10,338 rows


## 7. Resolve Head (Chemical) Names

1. PubChem IUPAC lookup.
2. Normalise DrugBank IDs; DrugBank name fallback.

In [41]:
# Step 1 – PubChem lookup
final_df_dedup['head_detail_name'] = final_df_dedup['head'].astype(str).map(Pubchem_IUPAC_CID_Dict)
final_df_dedup['head_smiles_name'] = final_df_dedup['head'].astype(str).map(Pubchem_CID_Smile_Dict)

# Step 2 – Normalise DrugBank IDs
def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df_dedup['head'] = final_df_dedup['head'].apply(standardize_drugbank_id).astype(str)

# DrugBank fallback
mask = final_df_dedup['head_detail_name'].isna() & final_df_dedup['head'].str.startswith('DB')
final_df_dedup.loc[mask, 'head_detail_name'] = final_df_dedup.loc[mask, 'head'].map(Drugbank_dict)

print(f"Missing head_detail_name: {final_df_dedup['head_detail_name'].isna().sum():,}")
print(f"Missing tail_detail_name: {final_df_dedup['tail_detail_name'].isna().sum():,}")

Missing head_detail_name: 0
Missing tail_detail_name: 0


## 8. Add Schema Columns and Enforce Column Order

In [42]:
EXTRA_COLS = ['head_smiles_name']
final_df_dedup = final_df_dedup[REQUIRED_COLS + EXTRA_COLS]

print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (10338, 14)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,1001,ChemicalEntity_Pathway,R-HSA-141333,ChemicalEntity,participates_in,Pathway,Monarch_KG,Pubchem,Reactome_ID,2-phenylethanamine,Biogenic amines are oxidatively deaminated to ...,Generalised,Homo species,C1=CC=C(C=C1)CCN
1,1001,ChemicalEntity_Pathway,R-HSA-375280,ChemicalEntity,participates_in,Pathway,Monarch_KG,Pubchem,Reactome_ID,2-phenylethanamine,Amine ligand-binding receptors,Generalised,Homo species,C1=CC=C(C=C1)CCN
2,1001,ChemicalEntity_Pathway,R-HSA-418555,ChemicalEntity,participates_in,Pathway,Monarch_KG,Pubchem,Reactome_ID,2-phenylethanamine,G alpha (s) signalling events,Generalised,Homo species,C1=CC=C(C=C1)CCN
3,10028772,ChemicalEntity_Pathway,R-HSA-156588,ChemicalEntity,participates_in,Pathway,Monarch_KG,Pubchem,Reactome_ID,"(2S,3S,4S,5R,6S)-3,4,5-trihydroxy-6-[(E)-6-(4-...",Glucuronidation,Generalised,Homo species,CC1=C2COC(=O)C2=C(C(=C1OC)C/C=C(\C)/CCC(=O)O[C...
4,10062737,ChemicalEntity_Pathway,R-HSA-5676934,ChemicalEntity,participates_in,Pathway,Monarch_KG,Pubchem,Reactome_ID,(2S)-2-amino-4-[(R)-methylsulfinyl]butanoic acid,Protein repair,Generalised,Homo species,C[S@@](=O)CC[C@@H](C(=O)O)N


## 9. QC — NaN Check

In [43]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,1670,0,1670
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [44]:
final_df_dedup[final_df_dedup['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name


## 10. Save Output

In [45]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 10,338 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_PATHWAY/ALL_CHEMICALENTITY_PATHWAY.csv


In [1]:
#